# Mini-projet final : fine-tuning de `bert-base-uncased` pour l'analyse de sentiment

**Scénario** : l'équipe support veut un signal de sentiment fiable sur des retours clients longs, pour escalader les clients mécontents avant qu'ils ne partent.

---

## Ce que je ne vais pas te laisser avaler sans discussion

Tu m'as demandé d'être sans complaisance. Voici les points réels — pas des chipotages — que j'ai trouvés dans cet énoncé, avec sources à l'appui quand j'ai vérifié :

1. **Bug silencieux confirmé dans `tf_encode`.** Le code original fait `list(encode_review(t).values())` puis assigne positionnellement `encoded[0]` à `input_ids`, `encoded[1]` à `attention_mask`, `encoded[2]` à `token_type_ids`. Or, l'ordre réel des clés retournées par `tokenizer.encode_plus()` est **`input_ids`, puis `token_type_ids`, puis `attention_mask`** (documenté par plusieurs sources indépendantes sur la structure du `BatchEncoding`). Le code original **inverse silencieusement `attention_mask` et `token_type_ids`** — pas d'erreur, pas de warning, juste un modèle entraîné sur des données corrompues. Je corrige ça en utilisant des clés explicites, pas l'ordre positionnel d'un dict.
2. **Risque réel et documenté avec `tf.keras.optimizers.Adam` sur des versions récentes de TensorFlow/Keras.** Sur certaines combinaisons keras 3 / tf_keras, `model.compile(optimizer=tf.keras.optimizers.Adam(...))` lève `ValueError: Could not interpret optimizer identifier` — un bug remonté publiquement sur GitHub en 2024. Ce n'est pas garanti de t'arriver (ça dépend de tes versions exactes), mais si ça t'arrive, ce n'est pas une erreur de ta part : essaie `tf.keras.optimizers.legacy.Adam` ou vérifie que `tf-keras` est bien installé en cohérence avec ta version de `transformers`.
3. **"Prediction: Positive (confidence=0.5)" dans l'énoncé original n'est PAS un résultat à célébrer, c'est un symptôme.** Pour une phrase clairement positive ("l'agent a poliment tout réglé") après un fine-tuning complet sur 25k exemples, un BERT correctement entraîné devrait output quelque chose comme 0.95-0.99, pas 0.5. Une confiance proche de 0.5 sur un exemple aussi peu ambigu est le signe d'un modèle **quasi non entraîné** — poids presque aléatoires côté tête de classification. Si tu observes ça après ton propre entraînement, ne te dis pas "c'est normal, l'énoncé l'a prévu" : diagnostique (bug de labels, learning rate, bug de tokenisation comme celui du point 1 — ce n'est probablement pas un hasard que ces deux problèmes coexistent dans le même énoncé).
4. **"Si vous voyez `GPU devices: []`, changez de runtime, aucune install supplémentaire n'est nécessaire sur Colab" est une simplification trompeuse.** Sur Colab, il faut *manuellement* aller dans `Runtime > Change runtime type > T4 GPU` **avant** d'exécuter quoi que ce soit — ce n'est pas automatique, et le changement de runtime redémarre ton kernel (tu perds toutes les variables déjà définies). L'énoncé donne l'impression que c'est fluide ; en pratique tu dois relancer tout le notebook depuis le début après le changement.
5. **Un score >90% d'accuracy sur IMDB n'est pas un exploit particulièrement impressionnant pour BERT fine-tuné.** C'est un score correct mais pas état de l'art — des fine-tunings soignés dépassent régulièrement 93-94% sur ce dataset. Ne présente pas 90% comme "la preuve que le fine-tuning est un succès total" dans tes conclusions ; c'est un seuil pédagogique raisonnable, pas un plafond de qualité.

Je n'ai pas exécuté ce notebook (pas de GPU ni d'accès au dataset IMDB via tfds dans mon environnement d'exécution). Le code suit les APIs documentées et corrige les bugs identifiés ci-dessus, mais teste-le toi-même avant de faire confiance aux résultats.

## Prérequis
- Python 3.9+
- Un runtime avec GPU (Colab, Kaggle, ou GPU local) idéalement, ~6 Go de VRAM libre. CPU fonctionne aussi mais l'entraînement sera beaucoup plus long.
- Packages : `tensorflow`, `tensorflow-datasets`, `transformers`, `accelerate`, `evaluate`.

In [ ]:
%pip install -q tensorflow tensorflow-datasets transformers accelerate evaluate

## Imports et vérification du matériel

In [ ]:
import platform
import tensorflow as tf
import tensorflow_datasets as tfds
from transformers import BertTokenizer, TFBertForSequenceClassification

print("Python version      :", platform.python_version())
print("TensorFlow version  :", tf.__version__)
print("GPU devices detected:", tf.config.list_physical_devices('GPU'))

**Si tu vois `GPU devices: []` sur Colab** : va dans `Runtime > Change runtime type`, choisis un GPU (T4), enregistre, puis **relance cette cellule depuis le début du notebook** (le changement de runtime redémarre le kernel, toutes tes variables précédentes sont perdues).

## Charger le dataset IMDB Reviews

In [ ]:
(ds_train, ds_test), ds_info = tfds.load(
    "imdb_reviews",
    split=(tfds.Split.TRAIN, tfds.Split.TEST),
    as_supervised=True,
    with_info=True
)
print(ds_info)

In [ ]:
for text, label in ds_train.take(2):
    print("Label:", "Positive" if label.numpy() else "Negative")
    print(text.numpy().decode()[:250], "...\n")

## Tokenizer et pipeline de données

In [ ]:
MAX_LENGTH = 256
BATCH_SIZE = 16

tokenizer = BertTokenizer.from_pretrained("bert-base-uncased", do_lower_case=True)
print("Tokenizer loaded:", tokenizer.name_or_path)

In [ ]:
def encode_review(review_input):
    if isinstance(review_input, bytes):
        review_text = review_input.decode("utf-8")
    elif hasattr(review_input, "numpy"):
        review_text = review_input.numpy().decode("utf-8")
    else:
        review_text = str(review_input)

    encoded = tokenizer.encode_plus(
        review_text,
        add_special_tokens=True,
        max_length=MAX_LENGTH,
        padding="max_length",
        truncation=True,
        return_attention_mask=True,
        return_token_type_ids=True,
    )

    # ⚠️ CORRECTIF : on retourne un tuple dans un ORDRE EXPLICITE ET CONTRÔLÉ PAR NOUS
    # (input_ids, attention_mask, token_type_ids), au lieu de faire confiance à l'ordre
    # interne du dict renvoyé par encode_plus() — cet ordre est en réalité
    # (input_ids, token_type_ids, attention_mask), ce qui inverserait silencieusement
    # attention_mask et token_type_ids si on les récupérait par position.
    return encoded["input_ids"], encoded["attention_mask"], encoded["token_type_ids"]


def tf_encode(text, label):
    input_ids, attention_mask, token_type_ids = tf.py_function(
        func=lambda t: encode_review(t),
        inp=[text],
        Tout=[tf.int32, tf.int32, tf.int32]
    )
    # tf.py_function perd les infos de shape statique : on les redéfinit explicitement
    # pour que .batch() plus bas fonctionne correctement.
    input_ids.set_shape([MAX_LENGTH])
    attention_mask.set_shape([MAX_LENGTH])
    token_type_ids.set_shape([MAX_LENGTH])

    return {
        "input_ids": input_ids,
        "attention_mask": attention_mask,
        "token_type_ids": token_type_ids
    }, label


def prepare_dataset(dataset, shuffle=True):
    ds = dataset.map(tf_encode, num_parallel_calls=tf.data.AUTOTUNE)
    if shuffle:
        ds = ds.shuffle(2000)
    return ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)


train_ds = prepare_dataset(ds_train, shuffle=True)
test_ds = prepare_dataset(ds_test, shuffle=False)  # pas besoin de shuffle sur le test

**Ce que j'ai changé par rapport à l'énoncé, et pourquoi ça compte** : shuffle sur le test set n'a aucune utilité et complique le débogage (tu ne peux plus aligner facilement prédictions et labels dans le même ordre pour une inspection manuelle) — je l'ai désactivé pour `test_ds`.

## Initialiser le modèle pour le fine-tuning

In [ ]:
model = TFBertForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=2,
    use_safetensors=False
)

optimizer = tf.keras.optimizers.Adam(learning_rate=2e-5, epsilon=1e-8)
loss_fn = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True)
metrics = [tf.keras.metrics.SparseCategoricalAccuracy(name="accuracy")]

model.compile(optimizer=optimizer, loss=loss_fn, metrics=metrics)
model.summary()

**Si cette cellule échoue avec `ValueError: Could not interpret optimizer identifier`** : c'est un conflit connu entre Keras 3 et `tf.keras.optimizers.Adam` sur certaines installations récentes. Essaie :
```python
optimizer = tf.keras.optimizers.legacy.Adam(learning_rate=2e-5, epsilon=1e-8)
```
ou vérifie que le package `tf-keras` est installé (`pip install tf-keras`) si tu utilises une version de `transformers` qui s'attend à l'API Keras 2.

## Entraîner et surveiller

In [ ]:
EPOCHS = 2  # augmente à 3 si le temps le permet

history = model.fit(
    train_ds,
    validation_data=test_ds,
    epochs=EPOCHS
)

**Ce que tu dois vraiment regarder** : si `accuracy` d'entraînement grimpe mais que `val_accuracy` stagne ou baisse dès l'epoch 2, tu sur-apprends — augmenter le nombre d'epochs ne va pas t'aider, il faudrait plutôt regarder le learning rate ou la régularisation. Ne te contente pas de lire le dernier chiffre affiché, regarde la courbe complète.

## Évaluer sur le jeu de test

In [ ]:
eval_metrics = model.evaluate(test_ds, return_dict=True)
print("Métriques sur le test set :", eval_metrics)

**Sur le seuil des 90%** : si tu es en dessous, ne suppose pas automatiquement "il faut plus d'epochs". Vérifie d'abord que le bug de tokenisation signalé plus haut (inversion attention_mask/token_type_ids) n'est pas la cause — c'est exactement le genre de bug qui plafonne discrètement les performances sans jamais planter. Si tu es à 90%+, rappelle-toi que ce n'est pas l'état de l'art sur ce dataset ; ne le vends pas comme un résultat exceptionnel dans un rapport pour de vrais stakeholders métier.

## Fonction d'inférence réutilisable

In [ ]:
def predict_sentiment(text: str):
    encoded = tokenizer(
        text,
        add_special_tokens=True,
        max_length=MAX_LENGTH,
        padding="max_length",
        truncation=True,
        return_tensors="tf"
    )

    outputs = model(encoded)
    logits = outputs.logits
    probs = tf.nn.softmax(logits, axis=-1).numpy()[0]

    predicted_class = int(probs.argmax())
    label = "Positive" if predicted_class == 1 else "Negative"

    return label, float(probs.max())


custom_sentence = "The onboarding emails were confusing, but the agent fixed everything politely."
label, confidence = predict_sentiment(custom_sentence)
print(f"Prediction: {label} (confidence={confidence:.3f})")

**Si tu obtiens une confiance proche de 0.5 ici**, ne recopie pas la remarque de l'énoncé original disant que c'est le résultat attendu. Ce n'est pas normal pour un modèle correctement fine-tuné sur un exemple aussi peu ambigu. Diagnostique dans cet ordre : (1) vérifie que le bug de tokenisation signalé plus haut est bien corrigé dans ton propre code, (2) vérifie que `history` montre une vraie baisse de la loss pendant l'entraînement, (3) vérifie que les labels du dataset ne sont pas inversés quelque part.

## Réflexion et prochaines étapes

**Pourquoi le fine-tuning est utile** : réutiliser un checkpoint public pour atteindre une bonne accuracy avec relativement peu de données propres à la tâche.

**Compétences transférables** : la même approche s'applique à de la classification de texte en RH, juridique, ou analytique produit.

**Ce que tu peux faire avec ça** : adaptation à un domaine spécifique (emails de ton entreprise), checkpoints multilingues (DistilBERT multilingue, XLM-R), monitoring (dérive des données, dashboards).

### Questions de réflexion (réponds-y honnêtement, pas pour la forme)

1. **Quel levier a le plus amélioré les résultats — nettoyage des données, hyperparamètres, plus d'epochs ?** Si tu n'as testé qu'une seule configuration, tu ne peux littéralement pas répondre à cette question de façon informée. Ne réponds pas par une intuition non testée présentée comme un fait.
2. **Où ajouterais-tu des garde-fous avant un déploiement en production ?** Pense au minimum à : un seuil de confiance en dessous duquel on escalade systématiquement à un humain plutôt que de faire confiance à la prédiction automatique, un mécanisme de détection de dérive (le vocabulaire des retours clients change avec le temps, un modèle entraîné sur IMDB — des critiques de films — n'a d'ailleurs aucune garantie de bien généraliser à de vrais tickets support, ce qui est un écart de domaine que l'exercice ne questionne jamais explicitement), et un audit régulier d'un échantillon de prédictions par un humain.
3. **Quelles parties prenantes bénéficient le plus (support lead, product manager, compliance officer) ?** Réponds en pensant aussi à qui porte le risque si le modèle se trompe sur un client à fort litige potentiel — ce n'est pas qu'une question de bénéfice, c'est aussi une question de responsabilité en cas d'erreur.